In [4]:
import numpy as np
import pandas as pd
import dai


def _safe_divide(a, b):
    return a / b.replace(0, np.nan)


def _cs_rank(df, col, ascending=True):
    values = df[col] if isinstance(col, str) else pd.Series(col, index=df.index)
    return values.groupby(df["date"]).transform(
        lambda x: x.rank(pct=True, ascending=ascending)
    )


def _cs_zscore(s, clip=3.0):
    mean = s.mean()
    std = s.std(ddof=0)
    if not np.isfinite(std) or std == 0:
        return pd.Series(0.0, index=s.index)
    return ((s - mean) / std).clip(-clip, clip)


def _fill_neutral_by_day(df, col):
    values = df[col] if isinstance(col, str) else pd.Series(col, index=df.index)
    return values.fillna(values.groupby(df["date"]).transform("median")).fillna(0.5)


def _clip01(s):
    return s.clip(lower=0.0, upper=1.0)


def main(datasources, start_date, end_date):
    bar1m = datasources.get("bar1m", "bigalpha_2026_stock_bar1m")

    start_d = str(start_date)[:10]
    end_d = str(end_date)[:10]
    adjusted_start = (
        pd.to_datetime(start_d) - pd.Timedelta(days=45)
    ).strftime("%Y-%m-%d")

    pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()
    pool["date"] = pd.to_datetime(pool["date"])
    pool["instrument"] = pool["instrument"].astype(str)

    # 分钟特征聚合为日频：关注尾盘承接、盘口改善、低冲击和早盘冲高后的修复。
    sql = f"""
    WITH base AS (
        SELECT
            date,
            date_trunc('day', date)::DATE::DATETIME AS day,
            instrument::string AS instrument,
            CAST(strftime(date, '%H%M%S') AS INTEGER) AS hhmmss,
            pre_close,
            high,
            close,
            amount,
            (bid_volume1 - ask_volume1)
                / NULLIF(bid_volume1 + ask_volume1, 0) AS book_imbalance,
            (
                bid_volume1 + bid_volume2 + bid_volume3 + bid_volume4 + bid_volume5
                - ask_volume1 - ask_volume2 - ask_volume3 - ask_volume4 - ask_volume5
            ) / NULLIF(
                bid_volume1 + bid_volume2 + bid_volume3 + bid_volume4 + bid_volume5
                + ask_volume1 + ask_volume2 + ask_volume3 + ask_volume4 + ask_volume5,
                0
            ) AS depth5_imbalance,
            (
                bid_volume1 + bid_volume2 + bid_volume3
                - ask_volume1 - ask_volume2 - ask_volume3
            ) / NULLIF(
                bid_volume1 + bid_volume2 + bid_volume3
                + ask_volume1 + ask_volume2 + ask_volume3,
                0
            ) AS buy_urgency
        FROM {bar1m}
    )
    SELECT
        day AS date,
        instrument,
        arg_max(close, date) AS close,
        arg_max(pre_close, date) AS pre_close,
        max(amount) AS daily_amount,
        max(CASE WHEN hhmmss < 130000 THEN amount END) AS amount_1300,
        max(CASE WHEN hhmmss < 143000 THEN amount END) AS amount_1430,
        max(CASE WHEN hhmmss >= 93000 AND hhmmss <= 100000 THEN high END) AS open_high,
        avg(CASE WHEN hhmmss >= 135500 AND hhmmss <= 140000 THEN close END) AS close_1400,
        avg(CASE WHEN hhmmss >= 142500 AND hhmmss < 143000 THEN close END) AS close_1430_ref,
        avg(CASE WHEN hhmmss >= 130000 AND hhmmss < 143000 THEN book_imbalance END)
            AS pre_tail_book_imbalance,
        avg(CASE WHEN hhmmss >= 143000 THEN book_imbalance END) AS tail_book_imbalance,
        avg(CASE WHEN hhmmss >= 130000 AND hhmmss < 143000 THEN depth5_imbalance END)
            AS pre_tail_depth5_imbalance,
        avg(CASE WHEN hhmmss >= 143000 THEN depth5_imbalance END) AS tail_depth5_imbalance,
        avg(CASE WHEN hhmmss >= 130000 AND hhmmss < 143000 THEN buy_urgency END)
            AS pre_tail_buy_urgency,
        avg(CASE WHEN hhmmss >= 143000 THEN buy_urgency END) AS tail_buy_urgency
    FROM base
    GROUP BY 1, 2
    ORDER BY 1, 2
    """
    raw = dai.query(
        sql,
        filters={"date": [adjusted_start + " 00:00:00", end_d + " 23:59:59"]},
        compression=True,
    ).df()
    raw["date"] = pd.to_datetime(raw["date"])
    raw["instrument"] = raw["instrument"].astype(str)
    raw = raw.sort_values(["instrument", "date"]).reset_index(drop=True)

    for col in [
        "close",
        "pre_close",
        "daily_amount",
        "amount_1300",
        "amount_1430",
        "open_high",
        "close_1400",
        "close_1430_ref",
        "pre_tail_book_imbalance",
        "tail_book_imbalance",
        "pre_tail_depth5_imbalance",
        "tail_depth5_imbalance",
        "pre_tail_buy_urgency",
        "tail_buy_urgency",
    ]:
        raw[col] = pd.to_numeric(raw[col], errors="coerce")

    raw["amount_1300"] = raw["amount_1300"].fillna(0.0)
    raw["amount_1430"] = raw["amount_1430"].fillna(raw["amount_1300"])
    raw["tail_amount"] = raw["daily_amount"] - raw["amount_1430"]
    raw["pre_tail_amount"] = raw["amount_1430"] - raw["amount_1300"]
    raw["afternoon_amount"] = raw["daily_amount"] - raw["amount_1300"]

    g = raw.groupby("instrument", group_keys=False)
    raw["afternoon_amount_ma20"] = g["afternoon_amount"].transform(
        lambda x: x.rolling(20, min_periods=8).mean().shift(1)
    )
    raw["tail_amount_ma20"] = g["tail_amount"].transform(
        lambda x: x.rolling(20, min_periods=8).mean().shift(1)
    )
    raw["ret_1d"] = g["close"].pct_change()
    raw["realized_vol20"] = g["ret_1d"].transform(
        lambda x: x.rolling(20, min_periods=8).std().shift(1)
    )

    raw["tail_amount_share"] = _safe_divide(raw["tail_amount"], raw["daily_amount"])
    raw["tail_amount_shock"] = _safe_divide(raw["tail_amount"], raw["tail_amount_ma20"])
    raw["afternoon_amount_shock"] = _safe_divide(
        raw["afternoon_amount"], raw["afternoon_amount_ma20"]
    )
    raw["tail_vs_pre_amount"] = _safe_divide(
        raw["tail_amount"], raw["pre_tail_amount"]
    )
    raw["book_improve"] = (
        raw["tail_book_imbalance"] - raw["pre_tail_book_imbalance"]
    )
    raw["depth_improve"] = (
        raw["tail_depth5_imbalance"] - raw["pre_tail_depth5_imbalance"]
    )
    raw["buy_urgency_improve"] = raw["tail_buy_urgency"] - raw["pre_tail_buy_urgency"]
    raw["close_1430_ref"] = raw["close_1430_ref"].fillna(raw["close"])
    raw["tail_ret"] = raw["close"] / raw["close_1430_ref"].replace(0, np.nan) - 1.0

    raw["open_spike"] = raw["open_high"] / raw["pre_close"].replace(0, np.nan) - 1.0
    raw["mid_pullback"] = raw["close_1400"] / raw["open_high"].replace(0, np.nan) - 1.0
    raw["repair_to_close"] = raw["close"] / raw["close_1400"].replace(0, np.nan) - 1.0
    raw["low_impact_raw"] = _safe_divide(
        raw["tail_amount_share"], raw["tail_ret"].abs() + 0.002
    )

    target = raw[
        (raw["date"] >= pd.Timestamp(start_d)) & (raw["date"] <= pd.Timestamp(end_d))
    ].copy()
    result = pool.merge(target, on=["date", "instrument"], how="left")

    feature_cols = [
        "tail_amount_share",
        "tail_amount_shock",
        "afternoon_amount_shock",
        "tail_vs_pre_amount",
        "book_improve",
        "depth_improve",
        "buy_urgency_improve",
        "low_impact_raw",
        "open_spike",
        "mid_pullback",
        "repair_to_close",
        "tail_ret",
        "realized_vol20",
    ]
    for col in feature_cols:
        result[col] = result[col].replace([np.inf, -np.inf], np.nan)

    # 全部转为日内截面分数，保证不同交易日可比，且平台后续中性化前的缺失率较低。
    result["tail_share_score"] = _fill_neutral_by_day(
        result, _cs_rank(result, "tail_amount_share")
    )
    result["tail_shock_score"] = _fill_neutral_by_day(
        result, _cs_rank(result, "tail_amount_shock")
    )
    result["afternoon_shock_score"] = _fill_neutral_by_day(
        result, _cs_rank(result, "afternoon_amount_shock")
    )
    result["tail_vs_pre_score"] = _fill_neutral_by_day(
        result, _cs_rank(result, "tail_vs_pre_amount")
    )
    result["book_score"] = _fill_neutral_by_day(result, _cs_rank(result, "book_improve"))
    result["depth_score"] = _fill_neutral_by_day(
        result, _cs_rank(result, "depth_improve")
    )
    result["urgency_score"] = _fill_neutral_by_day(
        result, _cs_rank(result, "buy_urgency_improve")
    )
    result["impact_score"] = _fill_neutral_by_day(
        result, _cs_rank(result, "low_impact_raw")
    )
    result["open_attention_score"] = _fill_neutral_by_day(
        result, _cs_rank(result, "open_spike")
    )
    result["pullback_score"] = _fill_neutral_by_day(
        result, _cs_rank(result, "mid_pullback", ascending=True)
    )
    result["repair_score"] = _fill_neutral_by_day(
        result, _cs_rank(result, "repair_to_close")
    )
    result["stable_vol_score"] = _fill_neutral_by_day(
        result, _cs_rank(result, "realized_vol20", ascending=False)
    )

    # 尾盘温和上涨最好；硬拉、急跌都扣分。
    result["tail_ret_ok"] = _clip01((result["tail_ret"] + 0.002) / 0.009).fillna(0.5)
    result["tail_abs_ret_score"] = _fill_neutral_by_day(
        result,
        _cs_rank(
            result,
            result["tail_ret"].abs().rename("tail_abs_ret"),
            ascending=False,
        ),
    )
    result["chase_penalty"] = _clip01((result["tail_ret"] - 0.012) / 0.018).fillna(0.0)

    result["late_absorption"] = (
        0.12 * result["tail_share_score"]
        + 0.12 * result["tail_shock_score"]
        + 0.08 * result["afternoon_shock_score"]
        + 0.08 * result["tail_vs_pre_score"]
        + 0.13 * result["book_score"]
        + 0.12 * result["depth_score"]
        + 0.08 * result["urgency_score"]
        + 0.13 * result["impact_score"]
        + 0.07 * result["tail_abs_ret_score"]
        + 0.07 * result["tail_ret_ok"]
    )

    # 早盘被关注、午后有消化、尾盘重新修复，刻画主题轮动里的低拥挤承接。
    result["intraday_repair"] = (
        0.35 * result["open_attention_score"]
        + 0.30 * result["pullback_score"]
        + 0.25 * result["repair_score"]
        + 0.10 * result["stable_vol_score"]
    )

    result["raw_factor"] = (
        0.72 * result["late_absorption"]
        + 0.28 * result["intraday_repair"]
        - 0.10 * result["chase_penalty"]
    )

    result["factor"] = result.groupby("date")["raw_factor"].transform(_cs_zscore)
    result["factor"] = result["factor"].replace([np.inf, -np.inf], np.nan)
    result["factor"] = result.groupby("date")["factor"].transform(
        lambda x: x.fillna(x.median())
    )
    result["factor"] = result["factor"].fillna(0.0)

    return result[["date", "instrument", "factor"]]


